# Style Transfer Evaluation — Genre LoRA Adapters

Evaluates how well each genre-specific LoRA adapter transfers narrative style
compared to the base model, using:

| Metric | What it measures |
|---|---|
| **Perplexity** | How confidently each adapter scores its own genre's text |
| **Embedding Clustering** | Do outputs from the same adapter cluster together in semantic space? |
| **Distinctiveness Score** | How different is each adapter's output from the base model and from other genres? |
| **Cross-Genre Perplexity** | Does a Horror adapter find Comedy text surprising (high PPL)? |
| **Side-by-Side Outputs** | Same prompt → all 4 adapters + base model |

**Models evaluated:**
```
Base   : meta-llama/Llama-3.2-3B-Instruct
Drama  : SatyaMoulika/llama-3.2-drama-lora
Horror : SatyaMoulika/llama-3.2-horror-lora
Sci-Fi : SatyaMoulika/llama-3.2-scifi-lora
Comedy : SatyaMoulika/llama-3.2-comedy-lora
```

## 1 · Install Dependencies

In [ ]:
!pip install -q transformers peft bitsandbytes accelerate datasets \
    sentence-transformers scikit-learn matplotlib seaborn umap-learn tqdm

## 2 · HuggingFace Login

In [ ]:
from huggingface_hub import login, whoami
login()
print('Logged in as:', whoami()['name'])

## 3 · Config

In [ ]:
HF_USERNAME  = 'SatyaMoulika'
BASE_MODEL   = 'meta-llama/Llama-3.2-3B-Instruct'

ADAPTERS = {
    'drama':  f'{HF_USERNAME}/llama-3.2-drama-lora',
    'horror': f'{HF_USERNAME}/llama-3.2-horror-lora',
    'scifi':  f'{HF_USERNAME}/llama-3.2-scifi-lora',
    'comedy': f'{HF_USERNAME}/llama-3.2-comedy-lora',
}

GENRE_COLORS = {
    'base':   '#888888',
    'drama':  '#4C9BE8',
    'horror': '#E84C4C',
    'scifi':  '#9B4CE8',
    'comedy': '#F4A645',
}

NUM_EVAL_SAMPLES = 10   # prompts to generate per adapter
MAX_NEW_TOKENS   = 250
TEMPERATURE      = 0.8
TOP_P            = 0.9

## 4 · Eval Prompts (same prompt set for all adapters)

In [ ]:
# Neutral prompts — no genre cues in the scene heading or context
# so stylistic differences come purely from the adapter
EVAL_PROMPTS = [
    {'scene_heading': 'INT. KITCHEN - MORNING',
     'context': 'Two people who have not spoken in years sit across from each other.'},
    {'scene_heading': 'EXT. EMPTY PARKING LOT - NIGHT',
     'context': 'A person waits alone under a flickering streetlight.'},
    {'scene_heading': 'INT. SMALL APARTMENT - DAY',
     'context': 'Someone receives an unexpected phone call.'},
    {'scene_heading': 'EXT. FOREST TRAIL - DUSK',
     'context': 'A group of people realise they have taken a wrong turn.'},
    {'scene_heading': 'INT. HOSPITAL WAITING ROOM - NIGHT',
     'context': 'A family waits for news they are not sure they want to hear.'},
    {'scene_heading': 'INT. OFFICE - DAY',
     'context': 'An employee is called into their manager\'s office unexpectedly.'},
    {'scene_heading': 'EXT. ROOFTOP - NIGHT',
     'context': 'Two strangers meet for the first time above a busy city.'},
    {'scene_heading': 'INT. CAR - MOVING - DAY',
     'context': 'A long silence breaks during a road trip.'},
    {'scene_heading': 'INT. SCHOOL HALLWAY - AFTERNOON',
     'context': 'A student discovers something they were not meant to find.'},
    {'scene_heading': 'EXT. BEACH - SUNRISE',
     'context': 'Someone buries something in the sand and walks away.'},
][:NUM_EVAL_SAMPLES]

def build_prompt(genre: str, scene_heading: str, context: str) -> str:
    return (
        f'### Instruction:\nWrite a {genre} screenplay scene.\n'
        f'Scene heading: {scene_heading}\n'
        f'Story context: {context}\n\n'
        f'### Screenplay:\n'
    )

print(f'{len(EVAL_PROMPTS)} eval prompts ready.')
print('\nSample prompt (Drama):')
print(build_prompt('drama', EVAL_PROMPTS[0]['scene_heading'], EVAL_PROMPTS[0]['context']))

## 5 · Model Loader Utility

In [ ]:
import torch
import gc
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
)

def load_model(adapter_repo: str = None):
    """
    Loads base model. If adapter_repo is given, attaches the LoRA adapter.
    Returns (model, tokenizer).
    """
    tok = AutoTokenizer.from_pretrained(BASE_MODEL)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        quantization_config=bnb_config,
        device_map='auto',
        torch_dtype=torch.float16,
    )

    if adapter_repo:
        model = PeftModel.from_pretrained(base, adapter_repo)
    else:
        model = base

    model.eval()
    return model, tok


def unload_model(model):
    del model
    gc.collect()
    torch.cuda.empty_cache()

## 6 · Generate Outputs for All Models

In [ ]:
from tqdm.auto import tqdm

def generate_scenes(model, tokenizer, genre: str) -> list[str]:
    scenes = []
    for p in tqdm(EVAL_PROMPTS, desc=genre):
        prompt = build_prompt(genre, p['scene_heading'], p['context'])
        inputs = tokenizer(prompt, return_tensors='pt', truncation=True).to(model.device)
        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=True,
                temperature=TEMPERATURE,
                top_p=TOP_P,
                pad_token_id=tokenizer.eos_token_id,
            )
        full  = tokenizer.decode(out[0], skip_special_tokens=True)
        scene = full.split('### Screenplay:')[-1].strip()
        scenes.append(scene)
    return scenes


# Collect all outputs — load/unload each model to manage VRAM
all_outputs = {}   # {model_name: [scene_str, ...]}

# Base model
print('\n── Base model ──')
model, tok = load_model(adapter_repo=None)
all_outputs['base'] = generate_scenes(model, tok, 'base')
unload_model(model)

# Genre adapters
for genre, adapter_repo in ADAPTERS.items():
    print(f'\n── {genre} adapter ──')
    model, tok = load_model(adapter_repo=adapter_repo)
    all_outputs[genre] = generate_scenes(model, tok, genre)
    unload_model(model)

print(f'\nGenerated {NUM_EVAL_SAMPLES} scenes x {len(all_outputs)} models.')

## 7 · Side-by-Side Output Inspection

In [ ]:
PREVIEW_N = 3   # how many prompts to display

for i in range(PREVIEW_N):
    p = EVAL_PROMPTS[i]
    print('='*65)
    print(f'PROMPT {i+1}: {p["scene_heading"]}')
    print(f'CONTEXT : {p["context"]}')
    print('='*65)
    for model_name, scenes in all_outputs.items():
        print(f'\n[{model_name.upper()}]')
        print(scenes[i][:400])
    print()

## 8 · Metric 1 — Perplexity per Model on Its Own Genre

Lower perplexity = model is more confident generating text in that style.
Fine-tuned adapter should have lower PPL on its own genre outputs vs the base model.

In [ ]:
import torch
import numpy as np

def compute_perplexity(model, tokenizer, texts: list[str]) -> float:
    """
    Computes mean perplexity of model over a list of text strings.
    Uses sliding window to handle texts longer than context window.
    """
    ppls = []
    model.eval()
    for text in texts:
        enc = tokenizer(text, return_tensors='pt', truncation=True,
                        max_length=512).to(model.device)
        input_ids = enc['input_ids']
        with torch.no_grad():
            loss = model(**enc, labels=input_ids).loss
        ppls.append(torch.exp(loss).item())
    return round(float(np.mean(ppls)), 2)


# For each genre adapter, compute PPL on its own generated outputs
# Compare against base model PPL on the same texts
ppl_results = {}   # {genre: {adapter_ppl, base_ppl}}

for genre, adapter_repo in ADAPTERS.items():
    texts = all_outputs[genre]

    # PPL from the fine-tuned adapter
    print(f'Computing PPL — {genre} adapter on {genre} outputs...')
    ft_model, ft_tok = load_model(adapter_repo=adapter_repo)
    ft_ppl = compute_perplexity(ft_model, ft_tok, texts)
    unload_model(ft_model)

    # PPL from base model on same texts
    print(f'Computing PPL — base model on {genre} outputs...')
    base_model, base_tok = load_model(adapter_repo=None)
    base_ppl = compute_perplexity(base_model, base_tok, texts)
    unload_model(base_model)

    ppl_results[genre] = {'adapter_ppl': ft_ppl, 'base_ppl': base_ppl}
    print(f'  {genre:8s} → adapter PPL: {ft_ppl:.2f}   base PPL: {base_ppl:.2f}\n')

print('Perplexity computation complete.')

## 9 · Metric 2 — Cross-Genre Perplexity Matrix

Each adapter scores every other genre's outputs.
High cross-genre PPL = adapters are stylistically distinct.
Low diagonal = each adapter is fluent in its own genre.

In [ ]:
import pandas as pd

genres     = list(ADAPTERS.keys())
ppl_matrix = pd.DataFrame(index=genres, columns=genres, dtype=float)

for eval_genre, adapter_repo in ADAPTERS.items():
    model, tok = load_model(adapter_repo=adapter_repo)

    for text_genre in genres:
        texts = all_outputs[text_genre]
        ppl   = compute_perplexity(model, tok, texts)
        ppl_matrix.loc[eval_genre, text_genre] = ppl
        print(f'  Adapter={eval_genre:8s}  TextGenre={text_genre:8s}  PPL={ppl:.2f}')

    unload_model(model)

print('\nCross-Genre Perplexity Matrix (rows=adapter, cols=text genre):')
print(ppl_matrix.to_string())

## 10 · Metric 3 — Distinctiveness Score

Cosine distance between each adapter's output embeddings and the base model's.
Higher = the adapter has shifted the generation further from the base model's default style.

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

st_model = SentenceTransformer('all-MiniLM-L6-v2')

def embed(texts: list[str]) -> np.ndarray:
    return st_model.encode(texts, show_progress_bar=False, normalize_embeddings=True)

base_embeddings = embed(all_outputs['base'])

distinctiveness = {}

for genre in ADAPTERS:
    genre_embeddings = embed(all_outputs[genre])
    # Cosine distance = 1 - cosine similarity (per sample, then averaged)
    sims = [
        cosine_similarity([base_embeddings[i]], [genre_embeddings[i]])[0][0]
        for i in range(len(EVAL_PROMPTS))
    ]
    dist_score = round(float(1 - np.mean(sims)), 4)
    distinctiveness[genre] = dist_score
    print(f'{genre:8s} distinctiveness from base: {dist_score:.4f}')

print('\nHigher score = outputs are more stylistically distant from the base model.')

## 11 · Metric 4 — Embedding Clustering (UMAP)

Reduces sentence embeddings to 2D. If adapters have captured distinct styles,
outputs from the same genre should cluster together.

In [ ]:
import umap
import matplotlib.pyplot as plt
import numpy as np

# Stack all embeddings with labels
all_texts  = []
all_labels = []

for model_name, scenes in all_outputs.items():
    all_texts.extend(scenes)
    all_labels.extend([model_name] * len(scenes))

all_embeddings = embed(all_texts)

# UMAP reduction to 2D
reducer   = umap.UMAP(n_components=2, random_state=42, n_neighbors=5, min_dist=0.3)
embedding_2d = reducer.fit_transform(all_embeddings)

# Plot
fig, ax = plt.subplots(figsize=(10, 7))

for model_name in all_outputs:
    mask  = [l == model_name for l in all_labels]
    pts   = embedding_2d[mask]
    color = GENRE_COLORS[model_name]
    ax.scatter(pts[:, 0], pts[:, 1], label=model_name.capitalize(),
               color=color, s=120, alpha=0.8, edgecolors='white', linewidth=0.5)

    # Centroid marker
    centroid = pts.mean(axis=0)
    ax.scatter(*centroid, color=color, s=300, marker='*',
               edgecolors='black', linewidth=1, zorder=5)

ax.set_title('UMAP Embedding Clusters — Genre LoRA Outputs vs Base Model',
             fontsize=13)
ax.set_xlabel('UMAP-1')
ax.set_ylabel('UMAP-2')
ax.legend(loc='best', fontsize=10)
ax.grid(alpha=0.2)

plt.tight_layout()
plt.savefig('umap_clusters.png', dpi=150, bbox_inches='tight')
plt.show()
print('UMAP plot saved.')

## 12 · Visualise — Perplexity Comparison Bar Chart

In [ ]:
import matplotlib.pyplot as plt

genres_list  = list(ppl_results.keys())
adapter_ppls = [ppl_results[g]['adapter_ppl'] for g in genres_list]
base_ppls    = [ppl_results[g]['base_ppl']    for g in genres_list]

x, width = range(len(genres_list)), 0.35
fig, ax  = plt.subplots(figsize=(10, 5))

b1 = ax.bar([i-width/2 for i in x], base_ppls,    width, label='Base model', color='#888888')
b2 = ax.bar([i+width/2 for i in x], adapter_ppls, width, label='Genre adapter',
            color=[GENRE_COLORS[g] for g in genres_list])

ax.set_xticks(list(x))
ax.set_xticklabels([g.capitalize() for g in genres_list])
ax.set_ylabel('Perplexity (lower = more fluent in genre)')
ax.set_title('Perplexity: Base Model vs Genre Adapter on Own Genre Outputs')
ax.legend()
ax.bar_label(b1, fmt='%.1f', padding=3, fontsize=8)
ax.bar_label(b2, fmt='%.1f', padding=3, fontsize=8)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('perplexity_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 13 · Visualise — Cross-Genre Perplexity Heatmap

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    ppl_matrix.astype(float),
    annot=True,
    fmt='.1f',
    cmap='RdYlGn_r',   # red = high PPL (confused), green = low PPL (confident)
    ax=ax,
    linewidths=0.5,
    square=True,
)
ax.set_title('Cross-Genre Perplexity Matrix\n(row=adapter, col=text genre)', fontsize=12)
ax.set_xlabel('Text Genre')
ax.set_ylabel('Adapter Used')

plt.tight_layout()
plt.savefig('cross_genre_ppl_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Heatmap saved.')

## 14 · Visualise — Distinctiveness Scores

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

bars = ax.bar(
    [g.capitalize() for g in distinctiveness],
    list(distinctiveness.values()),
    color=[GENRE_COLORS[g] for g in distinctiveness],
    width=0.5,
)
ax.bar_label(bars, fmt='%.4f', padding=3, fontsize=9)
ax.set_ylabel('Cosine Distance from Base Model')
ax.set_title('Distinctiveness Score — How Far Each Adapter Shifts Style from Base')
ax.set_ylim(0, max(distinctiveness.values()) * 1.3)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('distinctiveness_scores.png', dpi=150, bbox_inches='tight')
plt.show()

## 15 · Full Eval Summary Table

In [ ]:
import pandas as pd

summary_rows = []
for genre in ADAPTERS:
    summary_rows.append({
        'Genre':                   genre.capitalize(),
        'Adapter PPL (own genre)': ppl_results[genre]['adapter_ppl'],
        'Base PPL (own genre)':    ppl_results[genre]['base_ppl'],
        'PPL Delta':               round(
            ppl_results[genre]['base_ppl'] - ppl_results[genre]['adapter_ppl'], 2),
        'Distinctiveness':         distinctiveness[genre],
    })

summary_df = pd.DataFrame(summary_rows)
summary_df['Style Transfer?'] = summary_df['PPL Delta'].apply(
    lambda d: 'Yes ✅' if d > 0 else 'No ❌'
)

display(summary_df)
summary_df.to_csv('eval_summary.csv', index=False)
print('\nSaved eval_summary.csv')
print('\nInterpretation:')
print('  PPL Delta > 0  → adapter is more fluent in its genre than the base model')
print('  Distinctiveness → higher = outputs are more stylistically different from base')